# EPIC Clarity Condition Occurrence Hydration

This notebook hydrates the OMOP CONDITION_OCCURRENCE table from EPIC Clarity diagnosis data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PAT_ENC_DX` - Encounter diagnoses
- `_exponent._bronze_epic_clarity_*.dbo_PROBLEM_LIST` - Problem list (longitudinal diagnoses)

## OMOP Fields Populated
- condition_occurrence_id (surrogate key)
- condition_source_value
- condition_concept_id (mapped from diagnosis codes)
- condition_start_date
- visit_occurrence_id (if from encounter)

## Notes
- Diagnoses from both encounter diagnoses and problem list
- Concept mapping uses ICD9CM, ICD10CM, and SNOMED codes

In [ ]:
source = 'epic_clarity'

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW standard_diagnosis_mapping AS
WITH ranked AS (
  SELECT
    concept.vocabulary_id AS source_vocabulary_id,
    concept.concept_code AS source_concept_code,
    concept.concept_id AS source_concept_id,
    standard_concept.concept_id AS standard_concept_id,
    concept_relationship.valid_start_date AS rel_valid_start_date,
    ROW_NUMBER() OVER (
      PARTITION BY concept.vocabulary_id, concept.concept_code
      ORDER BY concept_relationship.valid_start_date DESC, standard_concept.concept_id ASC
    ) AS rn
  FROM _exponent.omop.concept
  JOIN _exponent.omop.concept_relationship
    ON concept.concept_id = concept_relationship.concept_id_1
    AND concept_relationship.relationship_id = 'Maps to'
    AND concept_relationship.invalid_reason IS NULL
  JOIN _exponent.omop.concept standard_concept
    ON concept_relationship.concept_id_2 = standard_concept.concept_id
    AND standard_concept.standard_concept_id = standard_concept.concept_id
  WHERE concept.standard_concept_id IS NULL
    AND concept.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
)
SELECT
  source_vocabulary_id,
  source_concept_code,
  source_concept_id,
  standard_concept_id
FROM ranked
WHERE rn = 1

In [ ]:
silver_condition_occurrence_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC_DX', 'PAT_ENC_CSN_ID', ped.PAT_ENC_CSN_ID) AS condition_source_value,
    ped.DX_ID_DX_NAME AS condition_code_source_value,
    ped.CONTACT_DATE AS condition_start_date,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', ped.PAT_ENC_CSN_ID) AS visit_occurrence_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.pat_enc_dx ped
WHERE ped.PAT_ENC_CSN_ID IS NOT NULL

UNION ALL

SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PROBLEM_LIST', 'PAT_ID', pl.PAT_ID) AS condition_source_value,
    pl.DX_ID_DX_NAME AS condition_code_source_value,
    pl.NOTED_DATE AS condition_start_date,
    NULL AS visit_occurrence_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.problem_list pl
WHERE pl.PAT_ID IS NOT NULL
''')

display(silver_condition_occurrence_df)
silver_condition_occurrence_df.createOrReplaceTempView("silver_condition_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.condition_occurrence AS target
USING silver_condition_occurrence AS source
ON target.condition_source_value = source.condition_source_value

WHEN MATCHED AND NOT (
    target.condition_code_source_value <=> source.condition_code_source_value
    AND target.condition_start_date <=> source.condition_start_date
    AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
)
THEN UPDATE SET
    target.condition_code_source_value = source.condition_code_source_value,
    target.condition_start_date = source.condition_start_date,
    target.visit_occurrence_source_value = source.visit_occurrence_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    condition_source_value,
    condition_code_source_value,
    condition_start_date,
    visit_occurrence_source_value,
    updated_tsp
)
VALUES (
    source.condition_source_value,
    source.condition_code_source_value,
    source.condition_start_date,
    source.visit_occurrence_source_value,
    source.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.condition_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, condition_source_value, updated_tsp
    FROM _exponent.omop_silver.condition_occurrence
    WHERE condition_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
    ON s.condition_source_value = x.condition_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
gold_condition_occurrence_df = spark.sql("""
SELECT
    m.condition_occurrence_id,
    s.condition_start_date,
    COALESCE(dcm.standard_concept_id, 0) AS condition_concept_id,
    mv.visit_occurrence_id,
    s.updated_tsp
FROM _exponent.omop_silver.condition_occurrence s
INNER JOIN _exponent.omop_mapping.source_to_condition_occurrence m
    ON s.condition_source_value = m.condition_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN standard_diagnosis_mapping dcm
    ON dcm.source_concept_code = s.condition_code_source_value
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence mv
    ON s.visit_occurrence_source_value = mv.visit_occurrence_source_value
    AND mv.source_system = 'epic_clarity'
    AND mv.active_flag = TRUE
""")

display(gold_condition_occurrence_df)
gold_condition_occurrence_df.createOrReplaceTempView("gold_condition_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop.condition_occurrence AS target
USING gold_condition_occurrence AS source
ON target.condition_occurrence_id = source.condition_occurrence_id

WHEN MATCHED AND NOT (
    target.condition_concept_id <=> source.condition_concept_id
    AND target.condition_start_date <=> source.condition_start_date
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
)
THEN UPDATE SET
    target.condition_concept_id = source.condition_concept_id,
    target.condition_start_date = source.condition_start_date,
    target.visit_occurrence_id = source.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
    condition_occurrence_id,
    condition_concept_id,
    condition_start_date,
    visit_occurrence_id
)
VALUES (
    source.condition_occurrence_id,
    source.condition_concept_id,
    source.condition_start_date,
    source.visit_occurrence_id
)